# Fine-tune Pegasus on SAMSum (Colab T4 GPU)

This notebook runs the **full** training (not smoke mode) on a free Colab GPU, then saves the model to Google Drive.

**Before you start:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → **Run all**.

| Step | What happens | Rough time |
|---|---|---|
| 1 | Check the GPU | seconds |
| 2 | Mount Google Drive (asks you to approve: stay for this) | 30 s |
| 3 | Clone the repo, install packages | 2–4 min |
| 4 | Optional low-memory settings (off by default) | seconds |
| 5 | Ingestion + tokenization | 1–2 min |
| 6 | Training + validation loss (downloads Pegasus, ~2.3 GB) | 60–90 min |
| 7 | Copy the model to Drive | 1–3 min |
| 8 | Summarize 2 test dialogues | 1 min |

Keep this tab open while it trains. Free Colab disconnects idle sessions.

Every pipeline command runs as `!python ...` in a separate process, followed by an `assert` on its exit code.
That way a failure **stops Run all** instead of silently continuing, and each process frees its GPU memory when it exits.

In [ ]:
REPO_URL = "https://github.com/s4nkalp47/text-summarizer.git"
DRIVE_DIR = "/content/drive/MyDrive/text-summarizer"

## 1. Check the GPU

`nvidia-smi` lists the GPU Colab gave you (expect **Tesla T4**, ~15 GB memory).
If PyTorch can't see a GPU, stop here and switch the runtime type: training Pegasus on a CPU would take days.

In [ ]:
!nvidia-smi
!python -c "import torch; assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > T4 GPU'; print('PyTorch', torch.__version__, '| GPU:', torch.cuda.get_device_name(0))"
assert _exit_code == 0, "No GPU available (see the message above)"

## 2. Mount Google Drive

A Colab machine is wiped when the session ends. Mounting Drive gives us somewhere permanent to save the trained model.
A pop-up asks you to approve access. It comes early on purpose, so you can approve it and then leave training running.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 3. Clone the repo and install packages

We install everything in `requirements.txt` **except `torch`**. Colab's preinstalled PyTorch is built for its own CUDA driver.
Replacing it would download ~2–3 GB and risk a driver mismatch. The project works with any recent PyTorch 2.x.

`pip install -e .` makes `import textSummarizer` work, exactly like on your laptop.
If Colab offers to "restart session" after the install, you can ignore it: the pipeline runs in fresh processes.

In [ ]:
import os

%cd /content
if os.path.isdir("text-summarizer"):
    !git -C text-summarizer pull
else:
    !git clone {REPO_URL}
assert _exit_code == 0, "git clone/pull failed"
%cd /content/text-summarizer
!git log --oneline -1

!grep -v "^torch==" requirements.txt > /tmp/requirements-colab.txt
!pip install -q -r /tmp/requirements-colab.txt
assert _exit_code == 0, "pip install -r requirements failed"
!pip install -q -e .
assert _exit_code == 0, "pip install -e . failed"
!python -c "import torch, transformers, datasets; print('torch', torch.__version__, '| transformers', transformers.__version__, '| datasets', datasets.__version__)"

## 4. CUDA out of memory? Low-memory settings

**Leave both switches off for the first run.** Use this cell only if step 6 crashes with `CUDA out of memory`. In that case:

1. Set `LOW_MEMORY = True`. This switches the optimizer to **Adafactor**, which stores far less state than AdamW,
   and turns on **gradient checkpointing**, which recomputes activations instead of storing them: ~20–30% slower, much less memory.
2. If it still runs out of memory, also set `MAX_INPUT_LENGTH = 384`. Dialogues are cut shorter, which saves activation memory.
   Step 5 re-tokenizes with the new length.
3. Click this cell, then **Runtime → Run cell and below**.

The edits only change `params.yaml` in this Colab copy, not your repo.

In [ ]:
import re
from pathlib import Path

LOW_MEMORY = False       # True: optim adafactor + gradient_checkpointing true
MAX_INPUT_LENGTH = None  # e.g. 384 if LOW_MEMORY alone isn't enough

params_file = Path("params.yaml")
params = params_file.read_text()
if LOW_MEMORY:
    params = re.sub(r"(optim:) \S+", r"\g<1> adafactor", params)
    params = re.sub(r"(gradient_checkpointing:) \S+", r"\g<1> true", params)
if MAX_INPUT_LENGTH:
    params = re.sub(r"(max_input_length:) \d+", rf"\g<1> {MAX_INPUT_LENGTH}", params)
params_file.write_text(params)

!grep -E "optim:|gradient_checkpointing:|max_input_length:" params.yaml

## 5. Ingestion and tokenization

- **Ingestion** downloads SAMSum (~10 MB) into `artifacts/data_ingestion/`.
- **Transformation** tokenizes every row with the **Pegasus** tokenizer.

`--no-smoke` overrides `smoke_test.enabled: true` in `params.yaml`, so the full data and the Pegasus checkpoint are used.

In [ ]:
!python main.py --stage ingestion --no-smoke
assert _exit_code == 0, "Ingestion failed (see the log above)"
!python main.py --stage transformation --no-smoke
assert _exit_code == 0, "Transformation failed (see the log above)"

## 6. Train

Fine-tunes `google/pegasus-cnn_dailymail` on the 14,731 training dialogues for 1 epoch:
batch size 1 × 16 gradient-accumulation steps = an effective batch of 16, so about **920 optimizer steps**.

What to watch:
- A `{'loss': ...}` line every 10 steps. It should **trend down**, roughly from ~3 toward ~1.5. It's noisy step to step; look at the trend.
- The learning rate climbs during the first 500 steps (warmup), then decays.
- At the end, the **validation loss** on 818 held-out dialogues is logged. It should be in the same range as the last training losses.

If this crashes with `CUDA out of memory`, go to step 4.

In [ ]:
!python main.py --stage training --no-smoke
assert _exit_code == 0, "Training failed. CUDA out of memory? See step 4"

## 7. Save the model to Google Drive

Copies `artifacts/model_trainer/` (model ~2.3 GB + tokenizer) to `MyDrive/text-summarizer/model_trainer`,
plus the run log (training and validation losses) to `MyDrive/text-summarizer/running_logs.log`.
Download the `model_trainer` folder from Drive to your laptop afterwards.

In [ ]:
import shutil

os.makedirs(DRIVE_DIR, exist_ok=True)
shutil.copytree("artifacts/model_trainer", f"{DRIVE_DIR}/model_trainer", dirs_exist_ok=True)
shutil.copy("logs/running_logs.log", f"{DRIVE_DIR}/running_logs.log")
!ls -lh {DRIVE_DIR}/model_trainer/model {DRIVE_DIR}/model_trainer/tokenizer

## 8. Sanity check: summarize 2 test dialogues

Loads the **fine-tuned** model from `artifacts/model_trainer/` and summarizes the first 2 test dialogues.
The model never saw these during training. It uses the generation settings from `params.yaml` (beam search with 8 beams).
Pegasus writes `<n>` for line breaks; we replace it with a space.

The summaries should be short, third-person, and name the people in the chat, like the human reference.

In [ ]:
%%writefile /content/sanity_check.py
import torch
from datasets import load_from_disk
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

from textSummarizer.config.configuration import ConfigurationManager

config = ConfigurationManager(smoke=False).get_model_evaluation_config()
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(config.tokenizer_path)
model = AutoModelForSeq2SeqLM.from_pretrained(config.model_path).to(device).eval()

for example in load_from_disk(config.data_path)["test"].select(range(2)):
    inputs = tokenizer(
        example["dialogue"], max_length=config.max_input_length, truncation=True, return_tensors="pt"
    ).to(device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            num_beams=config.num_beams,
            length_penalty=config.length_penalty,
            max_new_tokens=config.max_new_tokens,
        )
    summary = tokenizer.decode(output_ids[0], skip_special_tokens=True).replace("<n>", " ")
    print("=" * 80)
    print(example["dialogue"])
    print(f"\nREFERENCE: {example['summary']}")
    print(f"MODEL:     {summary}")

In [ ]:
!python /content/sanity_check.py
assert _exit_code == 0, "Sanity check failed (see the error above)"

## Done

1. In Google Drive, download `text-summarizer/model_trainer` (right-click → Download; Drive zips it).
2. Unzip it into your laptop's `artifacts/model_trainer/`, so you have `artifacts/model_trainer/model/` and `artifacts/model_trainer/tokenizer/`.
3. Optional: download `running_logs.log` to look at the loss curve.